# Power anomaly exploration

Read `power_anomaly_dataset_2022.csv` exactly as supplied. This table has 1,300 rows at **1 Hz**, not 1 kHz. Its reported timestamps are on **2025-10-10**, despite the filename. Timezone, measured/synthetic origin, asset level, voltage/current measurement type, and THD units have not been confirmed.

`Event_Label` and `Anomaly_Type` are source-supplied labels, not independently reviewed event annotations. This notebook preserves them for exploration; it does not add them to confirmed validation evidence. Original files are unchanged.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from di_sources import read_csv

frame = await read_csv("power_anomaly_dataset_2022.csv")
print(frame.shape)
print(frame.attrs)
frame.head()


## Inspect reported timing and label counts

Keep the source timestamp column. Parse a separate timezone-naive clock for within-file inspection; do not align it with AMI or other recordings until its timezone is known.


In [ ]:
clock = pd.to_datetime(frame["Timestamp"], errors="raise")
print("Reported clock:", clock.min(), "to", clock.max(), "| timezone:", clock.dt.tz)
print("Spacing (seconds):", clock.diff().dt.total_seconds().dropna().value_counts().to_dict())
print("Duplicate timestamps:", int(clock.duplicated().sum()))
print("Missing cells:", int(frame.isna().sum().sum()))
frame.groupby(["Event_Label", "Anomaly_Type"]).size().rename("samples").to_frame()


In [ ]:
# Each row below is a consecutive run of one source-supplied label.
groups = frame["Anomaly_Type"].ne(frame["Anomaly_Type"].shift()).cumsum()
runs = frame.assign(reported_clock=clock).groupby(groups).agg(
    label=("Anomaly_Type", "first"), first_sample=("reported_clock", "min"),
    last_sample=("reported_clock", "max"), samples=("Anomaly_Type", "size"))
runs


In [ ]:
elapsed = (clock - clock.iloc[0]).dt.total_seconds()
fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
for ax, column, unit in zip(axes, ["Load_kW", "Voltage", "Frequency_Hz"], ["kW", "source units", "Hz"]):
    ax.plot(elapsed, frame[column], linewidth=0.8)
    flagged = frame["Event_Label"].eq(1)
    ax.scatter(elapsed[flagged], frame.loc[flagged, column], s=5, color="#c46a29", label="Source anomaly label")
    ax.set_ylabel(f"{column} ({unit})")
    ax.grid(alpha=0.2)
axes[0].legend()
axes[-1].set_xlabel("Seconds from first reported timestamp - native 1 Hz samples")
fig.tight_layout()
plt.show()


## Try a custom anomaly score

This example uses the first 300 observations as a chronological development baseline and scores only subsequent observations. It does not fit thresholds to the later labels. The baseline must be reviewed before use; results from this single recording are exploratory, with no independent recording-level validation or sub-second timing claims.


In [ ]:
features = ["Voltage", "Current", "Load_kW", "Frequency_Hz"]
training = frame.iloc[:300][features]
center = training.median()
scale = (training - center).abs().median().replace(0, np.nan) * 1.4826
if scale.isna().any():
    raise ValueError("Choose nonconstant development features before scoring")
score = ((frame.loc[300:, features] - center).abs() / scale).max(axis=1)
threshold = 4.0  # Development setting: review its effect; no holdout tuning here.
review = frame.loc[score.index, ["Timestamp", "Event_Label", "Anomaly_Type"]].copy()
review["custom_score"] = score
review["screen_flag"] = score > threshold
print("Exploratory flagged samples:", int(review.screen_flag.sum()), "of", len(review))
review.sort_values("custom_score", ascending=False).head(20)


Change the score above to try your own algorithm. Preserve the supplied labels separately from model outputs. For Event Lab import, confirm timezone, source provenance, asset level, channel units/types, and event-label semantics first. Download this notebook as `.ipynb` to keep a portable copy.
